<a href="https://colab.research.google.com/github/kaluarachchiteshan/agromaster-risk-calculation-ml/blob/06_models_tuning/notebooks/06_models_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 06 - Train and tune the five classifiers on the validation folds (proposal 4.7) | Group 41 AgroMaster
# The final test period (block 6) is never used in this notebook.
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import ParameterSampler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

OUT_DIR = "/content/drive/MyDrive/agromaster"
RESULTS_FILE = os.path.join(OUT_DIR, "cv_results_models.csv")

# ---------------- CONFIG ----------------
TRAIN_SAMPLE = 400_000   # training rows sampled per fold for tuning (keeps run time practical on Colab)
SEED = 42
# ----------------------------------------
cfg = json.load(open(os.path.join(OUT_DIR, "feature_config.json")))
spl = json.load(open(os.path.join(OUT_DIR, "splits.json")))
NUM, CAT = cfg["num_features"], cfg["cat_features"]          # 15 numerical + 3 categorical = 18 features
N_BLOCKS, Q_HIGH, Q_MEDIUM = spl["n_blocks"], spl["q_high"], spl["q_medium"]

data = pd.read_parquet(os.path.join(OUT_DIR, "model_table.parquet"))
data = data[(data["block"] >= 0) & (data["block"] < N_BLOCKS)]     # development rows only
print(f"{len(data):,} development rows | {len(NUM) + len(CAT)} features")

# ----- helper functions (same as notebook 05) -----
def get_fold(data, k):
    """Fold k (1..5): train on blocks 0..k-1, validate on block k.
    Training rows whose 7-day target date reaches the validation period are excluded."""
    val = data[data["block"] == k]
    train = data[(data["block"] >= 0) & (data["block"] < k) & (data["target_date"] < val["date"].min())]
    return train, val

def fit_thresholds(train):
    """Commodity-specific 10th and 30th percentiles of the target ratio, estimated on training rows only."""
    thr = train.groupby("commodity", observed=True)["ratio"].quantile([Q_HIGH, Q_MEDIUM]).unstack()
    thr.columns = ["p_high", "p_medium"]
    return thr

def make_labels(d, thr, col="ratio"):
    """0 = Low, 1 = Medium, 2 = High risk, using the thresholds of each row's commodity."""
    p_high = d["commodity"].astype(str).map(thr["p_high"]).to_numpy(dtype="float64")
    p_med  = d["commodity"].astype(str).map(thr["p_medium"]).to_numpy(dtype="float64")
    r = d[col].to_numpy(dtype="float64")
    return np.where(r <= p_high, 2, np.where(r <= p_med, 1, 0)).astype("int8")

def score(y_true, y_pred):
    """Metrics from the proposal; macro F1 and High-risk recall are the main ones."""
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0),
            "high_recall": recall_score(y_true, y_pred, labels=[2], average="macro", zero_division=0),
            "high_precision": precision_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)}

Mounted at /content/drive
5,164,715 development rows | 18 features


In [ ]:
# Prepare each fold once: labels, class weights and two versions of the feature matrix.
#   "tree"   : median imputation + ordinal-encoded categories (Decision Tree, Random Forest, XGBoost)
#   "linear" : median imputation + standardisation + one-hot categories (Logistic Regression, Naive Bayes)
# Imputer, scaler, encoders, thresholds and class weights are all fitted on the training rows of the fold only.
folds = {}
for k in range(1, N_BLOCKS):
    train, val = get_fold(data, k)
    thr = fit_thresholds(train)                                   # thresholds from the full training fold
    if len(train) > TRAIN_SAMPLE:
        train = train.sample(TRAIN_SAMPLE, random_state=SEED)     # subsample only for model fitting
    y_tr, y_va = make_labels(train, thr), make_labels(val, thr)

    tree_prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUM),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)])
    lin_prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    folds[k] = {
        "tree":   (tree_prep.fit_transform(train).astype("float32"), tree_prep.transform(val).astype("float32")),
        "linear": (lin_prep.fit_transform(train).astype("float32"),  lin_prep.transform(val).astype("float32")),
        "y": (y_tr, y_va),
        "w": compute_sample_weight("balanced", y_tr),             # class weights: rare High class counts more
    }
    print(f"Fold {k}: train {len(y_tr):,} | val {len(y_va):,} | train classes Low/Med/High = "
          + "/".join(f"{(y_tr == c).mean():.2f}" for c in [0, 1, 2]))
del data

def run_cv(name, kind, build, grid, n_iter):
    """Randomised search: try n_iter parameter settings on all 5 folds, save and print mean validation scores."""
    rows = []
    for params in ParameterSampler(grid, n_iter=n_iter, random_state=SEED):
        t0 = time.time()
        for k, f in folds.items():
            (X_tr, X_va), (y_tr, y_va) = f[kind], f["y"]
            pred_tr, pred_va, extra = build(params, X_tr, y_tr, X_va, y_va, f["w"])
            rows.append({"model": name, "params": json.dumps(params, sort_keys=True), "fold": k, **score(y_va, pred_va),
                         "train_macro_f1": score(y_tr, pred_tr)["macro_f1"], **extra})
        last = pd.DataFrame(rows[-len(folds):])
        print(f"{params} -> macro F1 {last['macro_f1'].mean():.3f} | High recall {last['high_recall'].mean():.3f} "
              f"| train macro F1 {last['train_macro_f1'].mean():.3f} | {time.time() - t0:.0f}s")
        # save after every setting so nothing is lost if the Colab session disconnects
        old = pd.read_csv(RESULTS_FILE) if os.path.exists(RESULTS_FILE) else pd.DataFrame()
        old = old[old["model"] != name] if len(old) else old
        pd.concat([old, pd.DataFrame(rows)], ignore_index=True).to_csv(RESULTS_FILE, index=False)
    res = pd.DataFrame(rows).groupby("params")[["macro_f1", "high_recall"]].mean()
    print(f"Best {name}: {res['macro_f1'].idxmax()} (mean macro F1 {res['macro_f1'].max():.3f})\n")

def sk_build(make_model):
    """Wrap a scikit-learn model so that run_cv can fit it and get train / validation predictions."""
    def build(params, X_tr, y_tr, X_va, y_va, w):
        model = make_model(params).fit(X_tr, y_tr)
        return model.predict(X_tr), model.predict(X_va), {}
    return build

Fold 1: train 400,000 | val 860,784 | train classes Low/Med/High = 0.70/0.20/0.10
Fold 2: train 400,000 | val 860,608 | train classes Low/Med/High = 0.70/0.20/0.10
Fold 3: train 400,000 | val 860,242 | train classes Low/Med/High = 0.70/0.20/0.10
Fold 4: train 400,000 | val 861,870 | train classes Low/Med/High = 0.70/0.20/0.10
Fold 5: train 400,000 | val 860,821 | train classes Low/Med/High = 0.70/0.20/0.10


In [ ]:
# Model 1: Gaussian Naive Bayes (equal class priors play the role of class weights)
run_cv("Gaussian Naive Bayes", "linear",
       sk_build(lambda p: GaussianNB(priors=[1/3, 1/3, 1/3], **p)),
       {"var_smoothing": [1e-9, 1e-6, 1e-3]}, n_iter=3)

# Model 2: regularised multinomial Logistic Regression (C = inverse regularisation strength)
run_cv("Logistic Regression", "linear",
       sk_build(lambda p: LogisticRegression(class_weight="balanced", max_iter=300, **p)),
       {"C": [0.01, 0.1, 1.0]}, n_iter=3)

{'var_smoothing': 1e-09} -> macro F1 0.361 | High recall 0.676 | train macro F1 0.285 | 19s
{'var_smoothing': 1e-06} -> macro F1 0.369 | High recall 0.672 | train macro F1 0.293 | 16s
{'var_smoothing': 0.001} -> macro F1 0.408 | High recall 0.588 | train macro F1 0.341 | 16s
Best Gaussian Naive Bayes: {"var_smoothing": 0.001} (mean macro F1 0.408)

{'C': 0.01} -> macro F1 0.543 | High recall 0.716 | train macro F1 0.548 | 119s
{'C': 0.1} -> macro F1 0.544 | High recall 0.715 | train macro F1 0.549 | 130s
{'C': 1.0} -> macro F1 0.544 | High recall 0.715 | train macro F1 0.549 | 136s
Best Logistic Regression: {"C": 0.1} (mean macro F1 0.544)



In [ ]:
# Model 3: Decision Tree
run_cv("Decision Tree", "tree",
       sk_build(lambda p: DecisionTreeClassifier(class_weight="balanced", random_state=SEED, **p)),
       {"max_depth": [6, 10, 14, 18], "min_samples_leaf": [20, 100, 500]}, n_iter=6)

{'min_samples_leaf': 100, 'max_depth': 18} -> macro F1 0.498 | High recall 0.563 | train macro F1 0.609 | 43s
{'min_samples_leaf': 20, 'max_depth': 18} -> macro F1 0.487 | High recall 0.543 | train macro F1 0.674 | 45s
{'min_samples_leaf': 20, 'max_depth': 6} -> macro F1 0.471 | High recall 0.532 | train macro F1 0.505 | 22s
{'min_samples_leaf': 500, 'max_depth': 14} -> macro F1 0.497 | High recall 0.553 | train macro F1 0.565 | 32s
{'min_samples_leaf': 500, 'max_depth': 10} -> macro F1 0.495 | High recall 0.554 | train macro F1 0.559 | 30s
{'min_samples_leaf': 500, 'max_depth': 6} -> macro F1 0.472 | High recall 0.534 | train macro F1 0.507 | 21s
Best Decision Tree: {"max_depth": 18, "min_samples_leaf": 100} (mean macro F1 0.498)



In [ ]:
# Model 4: Random Forest (each tree sees a bootstrap sample of 100,000 rows to keep it fast)
run_cv("Random Forest", "tree",
       sk_build(lambda p: RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", max_samples=100_000,
                                                 n_jobs=-1, random_state=SEED, **p)),
       {"max_depth": [10, 16, 22], "min_samples_leaf": [5, 20, 100], "max_features": ["sqrt", 0.5]}, n_iter=4)

{'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 10} -> macro F1 0.564 | High recall 0.573 | train macro F1 0.621 | 209s
{'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 10} -> macro F1 0.561 | High recall 0.588 | train macro F1 0.614 | 201s
{'min_samples_leaf': 100, 'max_features': 'sqrt', 'max_depth': 16} -> macro F1 0.556 | High recall 0.605 | train macro F1 0.603 | 211s
{'min_samples_leaf': 100, 'max_features': 0.5, 'max_depth': 10} -> macro F1 0.555 | High recall 0.597 | train macro F1 0.605 | 322s
Best Random Forest: {"max_depth": 10, "max_features": "sqrt", "min_samples_leaf": 5} (mean macro F1 0.564)



In [ ]:
# Model 5: XGBoost (gradient boosting) with early stopping on the validation fold
from xgboost import XGBClassifier

def xgb_build(params, X_tr, y_tr, X_va, y_va, w):
    model = XGBClassifier(n_estimators=400, tree_method="hist", objective="multi:softprob", eval_metric="mlogloss",
                          early_stopping_rounds=30, n_jobs=-1, random_state=SEED, **params)
    model.fit(X_tr, y_tr, sample_weight=w, eval_set=[(X_va, y_va)], verbose=False)
    return model.predict(X_tr), model.predict(X_va), {"best_iteration": int(model.best_iteration)}

run_cv("XGBoost", "tree", xgb_build,
       {"learning_rate": [0.05, 0.1, 0.2], "max_depth": [4, 6, 8], "subsample": [0.7, 1.0],
        "colsample_bytree": [0.7, 1.0], "reg_lambda": [1.0, 5.0]}, n_iter=5)

{'subsample': 0.7, 'reg_lambda': 1.0, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.7} -> macro F1 0.563 | High recall 0.547 | train macro F1 0.646 | 566s
{'subsample': 0.7, 'reg_lambda': 5.0, 'max_depth': 4, 'learning_rate': 0.2, 'colsample_bytree': 1.0} -> macro F1 0.554 | High recall 0.544 | train macro F1 0.626 | 293s
{'subsample': 0.7, 'reg_lambda': 5.0, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.7} -> macro F1 0.562 | High recall 0.540 | train macro F1 0.647 | 369s
{'subsample': 0.7, 'reg_lambda': 1.0, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.7} -> macro F1 0.559 | High recall 0.553 | train macro F1 0.610 | 557s
{'subsample': 0.7, 'reg_lambda': 1.0, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.7} -> macro F1 0.561 | High recall 0.537 | train macro F1 0.651 | 244s
Best XGBoost: {"colsample_bytree": 0.7, "learning_rate": 0.05, "max_depth": 6, "reg_lambda": 1.0, "subsample": 0.7} (mean macro F1 0.563)



In [ ]:
# Summary: best setting of each model, next to the two baselines
res = pd.read_csv(RESULTS_FILE)
mean = res.groupby(["model", "params"])[["accuracy", "macro_f1", "high_recall", "high_precision", "train_macro_f1"]].mean().reset_index()
best = mean.loc[mean.groupby("model")["macro_f1"].idxmax()]
base = pd.read_csv(os.path.join(OUT_DIR, "cv_results_baselines.csv")).groupby("model")[["accuracy", "macro_f1", "high_recall", "high_precision"]].mean().reset_index()
table = pd.concat([base, best.drop(columns="params")], ignore_index=True).sort_values("macro_f1", ascending=False)
table["overfit_gap"] = table["train_macro_f1"] - table["macro_f1"]       # large gap = overfitting
print("Mean over the 5 validation folds (best setting per model):\n" + table.round(3).to_string(index=False))
print("\nBest parameters:")
for r in best.itertuples(): print(f"  {r.model}: {r.params}")
print("\nMacro F1 per fold (stability over time):")
bp = res.merge(best[["model", "params"]], on=["model", "params"])
print(bp.pivot(index="model", columns="fold", values="macro_f1").round(3).to_string())

table.to_csv(os.path.join(OUT_DIR, "cv_summary.csv"), index=False)
json.dump({r.model: json.loads(r.params) for r in best.itertuples()}, open(os.path.join(OUT_DIR, "best_params.json"), "w"), indent=2)
print("\nSaved cv_summary.csv and best_params.json")

Mean over the 5 validation folds (best setting per model):
               model  accuracy  macro_f1  high_recall  high_precision  train_macro_f1  overfit_gap
    Risk persistence     0.833     0.739        0.644           0.758             NaN          NaN
       Random Forest     0.695     0.564        0.573           0.419           0.621        0.057
             XGBoost     0.695     0.563        0.547           0.457           0.646        0.082
 Logistic Regression     0.651     0.544        0.715           0.372           0.549        0.005
       Decision Tree     0.604     0.498        0.563           0.318           0.609        0.111
Gaussian Naive Bayes     0.516     0.408        0.588           0.216           0.341       -0.066
      Majority class     0.704     0.275        0.000           0.000             NaN          NaN

Best parameters:
  Decision Tree: {"max_depth": 18, "min_samples_leaf": 100}
  Gaussian Naive Bayes: {"var_smoothing": 0.001}
  Logistic Regression: